# CSV Creation — Document-Level Dataset

This notebook creates document-level CSV files from the Markdown corpus.

It reads:

- `markdown/cleaned/policy/`
- `markdown/cleaned/sentiment/`
- `markdown/synthetic/sentiment/mds/`

It writes:

- `csvs/new/policy/policy_documents_full.csv`
- `csvs/new/sentiment/sentiment_documents_full.csv`
- `csvs/raw/synthetic/sentiment/synthetic_sentiment_documents_full.csv`
- `csvs/raw/documents_full_all.csv`
- `csvs/raw/documents_full_summary.csv`

For now, there is **no chunking**.  
Each Markdown document becomes one row in the CSV.

In [ ]:
from pathlib import Path
import re
import pandas as pd
from datetime import datetime

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    """
    Find the project root by walking upward until a folder containing
    both markdown/ and csvs/ is found.
    """
    if start is None:
        start = Path.cwd()

    start = start.resolve()

    for candidate in [start] + list(start.parents):
        if (candidate / "markdown").exists() and (candidate / "csvs").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root. Run this notebook from inside MSC_RESEARCH_PROJECT, "
        "or make sure the project contains both markdown/ and csvs/ folders."
    )


PROJECT_ROOT = find_project_root()

print("Project root:", PROJECT_ROOT)

In [ ]:
# Input folders
POLICY_MD_DIR = PROJECT_ROOT / "markdown" / "cleaned" / "policy"
SENTIMENT_MD_DIR = PROJECT_ROOT / "markdown" / "cleaned" / "sentiment"
SYNTHETIC_SENTIMENT_MD_DIR = PROJECT_ROOT / "markdown" / "synthetic" / "sentiment" / "mds"

# Output folders
CSVS_NEW_DIR = PROJECT_ROOT / "csvs" / "new"
POLICY_OUT_DIR = CSVS_NEW_DIR / "policy"
SENTIMENT_OUT_DIR = CSVS_NEW_DIR / "sentiment"
SYNTHETIC_SENTIMENT_OUT_DIR = CSVS_NEW_DIR / "synthetic" / "sentiment"

for folder in [
    CSVS_NEW_DIR,
    POLICY_OUT_DIR,
    SENTIMENT_OUT_DIR,
    SYNTHETIC_SENTIMENT_OUT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Policy Markdown folder:", POLICY_MD_DIR)
print("Original sentiment Markdown folder:", SENTIMENT_MD_DIR)
print("Synthetic sentiment Markdown folder:", SYNTHETIC_SENTIMENT_MD_DIR)
print("Output folder:", CSVS_NEW_DIR)

In [ ]:
def list_markdown_files(folder: Path) -> list[Path]:
    """
    List Markdown files recursively.
    Hidden/system files are ignored.
    """
    if not folder.exists():
        print(f"Warning: folder does not exist: {folder}")
        return []

    files = sorted(folder.rglob("*.md"))

    files = [
        path for path in files
        if not any(part.startswith(".") for part in path.parts)
    ]

    return files


policy_files = list_markdown_files(POLICY_MD_DIR)
sentiment_files = list_markdown_files(SENTIMENT_MD_DIR)
synthetic_sentiment_files = list_markdown_files(SYNTHETIC_SENTIMENT_MD_DIR)

print("Policy files:", len(policy_files))
print("Original sentiment files:", len(sentiment_files))
print("Synthetic sentiment files:", len(synthetic_sentiment_files))

print("\nExamples:")
for path in policy_files[:3]:
    print("POLICY:", path.relative_to(PROJECT_ROOT))

for path in sentiment_files[:3]:
    print("SENTIMENT:", path.relative_to(PROJECT_ROOT))

for path in synthetic_sentiment_files[:3]:
    print("SYNTHETIC:", path.relative_to(PROJECT_ROOT))

In [ ]:
def normalize_document_text(text: str) -> str:
    """
    Light normalization for document-level CSV.

    This does not chunk the document and does not remove meaningful content.
    It only removes remaining technical Markdown artifacts if they exist.
    """
    # Remove HTML comments such as image metadata if any remain.
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)

    # Remove Markdown image links if any remain.
    text = re.sub(r"\\?!\[[^\]]*\]\([^)]+\)", " ", text)

    # Remove raw local image paths if any remain.
    text = re.sub(
        r"/home/[^\s)]+?\.(png|jpg|jpeg|webp)",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # Remove code fences but keep any inner text.
    text = re.sub(r"```[\w-]*", " ", text)

    # Normalize non-breaking spaces.
    text = text.replace("\xa0", " ")

    # Normalize line endings.
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Collapse spaces inside each line.
    lines = [
        re.sub(r"[ \t]+", " ", line).strip()
        for line in text.split("\n")
    ]

    # Remove excessive blank lines.
    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def count_words(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text))


def infer_country_from_policy_path(path: Path) -> str:
    """
    Infer country from:
    markdown/cleaned/policy/<country>/file.md
    """
    try:
        rel = path.relative_to(POLICY_MD_DIR)

        if len(rel.parts) >= 2:
            return rel.parts[0]

    except ValueError:
        pass

    return ""


def infer_synthetic_type(filename: str) -> str:
    """
    Infer synthetic type from filenames such as:

    sentiment_comment_001.md
    sentiment_institutional_statement_001.md
    sentiment_report_summary_001.md
    """
    name = filename.lower()

    if "institutional_statement" in name:
        return "institutional_statement"

    if "report_summary" in name:
        return "report_summary"

    if "comment" in name:
        return "comment"

    return ""


def make_doc_id(prefix: str, path: Path) -> str:
    """
    Create a stable document id from prefix and file stem.
    """
    stem = re.sub(r"[^a-zA-Z0-9]+", "_", path.stem)
    stem = stem.strip("_").lower()

    return f"{prefix}_{stem}"

In [ ]:
def build_records_for_files(
    files: list[Path],
    *,
    corpus: str,
    source_type: str,
    base_dir: Path,
    doc_prefix: str,
) -> list[dict]:
    records = []

    for path in files:
        raw_text = path.read_text(encoding="utf-8", errors="replace")
        text = normalize_document_text(raw_text)

        if not text:
            print(f"Warning: empty text after normalization: {path.relative_to(PROJECT_ROOT)}")
            continue

        country = ""
        synthetic_type = ""

        if corpus == "policy":
            country = infer_country_from_policy_path(path)

        if source_type == "synthetic":
            country = "synthetic_sentiment"
            synthetic_type = infer_synthetic_type(path.name)

        try:
            source_file = str(path.relative_to(PROJECT_ROOT))
        except ValueError:
            source_file = str(path)

        try:
            relative_source_file = str(path.relative_to(base_dir))
        except ValueError:
            relative_source_file = path.name

        records.append(
            {
                "doc_id": make_doc_id(doc_prefix, path),
                "filename": path.name,
                "source_file": source_file,
                "relative_source_file": relative_source_file,
                "corpus": corpus,
                "source_type": source_type,
                "synthetic_type": synthetic_type,
                "country": country,
                "text": text,
                "char_count": len(text),
                "word_count": count_words(text),
            }
        )

    return records


policy_records = build_records_for_files(
    policy_files,
    corpus="policy",
    source_type="original",
    base_dir=POLICY_MD_DIR,
    doc_prefix="policy",
)

sentiment_records = build_records_for_files(
    sentiment_files,
    corpus="sentiment",
    source_type="original",
    base_dir=SENTIMENT_MD_DIR,
    doc_prefix="sentiment_original",
)

synthetic_sentiment_records = build_records_for_files(
    synthetic_sentiment_files,
    corpus="sentiment",
    source_type="synthetic",
    base_dir=SYNTHETIC_SENTIMENT_MD_DIR,
    doc_prefix="sentiment_synthetic",
)

print("Policy records:", len(policy_records))
print("Original sentiment records:", len(sentiment_records))
print("Synthetic sentiment records:", len(synthetic_sentiment_records))

In [ ]:
df_policy = pd.DataFrame(policy_records)
df_sentiment = pd.DataFrame(sentiment_records)
df_synthetic_sentiment = pd.DataFrame(synthetic_sentiment_records)

df_all = pd.concat(
    [
        df_policy,
        df_sentiment,
        df_synthetic_sentiment,
    ],
    ignore_index=True,
)

print("Policy dataframe:", df_policy.shape)
print("Original sentiment dataframe:", df_sentiment.shape)
print("Synthetic sentiment dataframe:", df_synthetic_sentiment.shape)
print("All dataframe:", df_all.shape)

df_all.head()

In [ ]:
policy_csv = POLICY_OUT_DIR / "policy_documents_full.csv"
sentiment_csv = SENTIMENT_OUT_DIR / "sentiment_documents_full.csv"
synthetic_sentiment_csv = SYNTHETIC_SENTIMENT_OUT_DIR / "synthetic_sentiment_documents_full.csv"
all_csv = CSVS_NEW_DIR / "documents_full_all.csv"

df_policy.to_csv(policy_csv, index=False, encoding="utf-8")
df_sentiment.to_csv(sentiment_csv, index=False, encoding="utf-8")
df_synthetic_sentiment.to_csv(synthetic_sentiment_csv, index=False, encoding="utf-8")
df_all.to_csv(all_csv, index=False, encoding="utf-8")

print("Saved:")
print("-", policy_csv.relative_to(PROJECT_ROOT))
print("-", sentiment_csv.relative_to(PROJECT_ROOT))
print("-", synthetic_sentiment_csv.relative_to(PROJECT_ROOT))
print("-", all_csv.relative_to(PROJECT_ROOT))

In [ ]:
def summarize_dataframe(df: pd.DataFrame, name: str):
    print(f"\n{name}")
    print("-" * len(name))

    print("Documents:", len(df))

    if len(df) == 0:
        print("Total characters: 0")
        print("Total words: 0")
        return

    print("Total characters:", int(df["char_count"].sum()))
    print("Total words:", int(df["word_count"].sum()))
    print("Average characters per document:", round(df["char_count"].mean(), 2))
    print("Average words per document:", round(df["word_count"].mean(), 2))


summarize_dataframe(df_policy, "Policy")
summarize_dataframe(df_sentiment, "Original sentiment")
summarize_dataframe(df_synthetic_sentiment, "Synthetic sentiment")
summarize_dataframe(df_all, "All documents")

In [ ]:
summary_by_source = (
    df_all
    .groupby(
        [
            "corpus",
            "source_type",
            "synthetic_type",
        ],
        dropna=False,
    )
    .agg(
        documents=("doc_id", "count"),
        total_characters=("char_count", "sum"),
        total_words=("word_count", "sum"),
        avg_characters=("char_count", "mean"),
        avg_words=("word_count", "mean"),
    )
    .reset_index()
)

summary_by_source

In [ ]:
summary_csv = CSVS_NEW_DIR / "documents_full_summary.csv"
summary_by_source.to_csv(summary_csv, index=False, encoding="utf-8")

print("Saved summary:")
print(summary_csv.relative_to(PROJECT_ROOT))

In [ ]:
balance = (
    df_all
    .groupby(
        [
            "corpus",
            "source_type",
        ],
        dropna=False,
    )
    .agg(
        documents=("doc_id", "count"),
        total_characters=("char_count", "sum"),
        total_words=("word_count", "sum"),
    )
    .reset_index()
)

balance

In [ ]:
policy_chars = df_policy["char_count"].sum() if len(df_policy) else 0
original_sentiment_chars = df_sentiment["char_count"].sum() if len(df_sentiment) else 0
synthetic_sentiment_chars = df_synthetic_sentiment["char_count"].sum() if len(df_synthetic_sentiment) else 0
combined_sentiment_chars = original_sentiment_chars + synthetic_sentiment_chars

print("Policy characters:", policy_chars)
print("Original sentiment characters:", original_sentiment_chars)
print("Synthetic sentiment characters:", synthetic_sentiment_chars)
print("Combined sentiment characters:", combined_sentiment_chars)
print("Difference policy - combined sentiment:", policy_chars - combined_sentiment_chars)

In [ ]:
preview_columns = [
    "doc_id",
    "filename",
    "corpus",
    "source_type",
    "synthetic_type",
    "country",
    "char_count",
    "word_count",
]

df_all[preview_columns].head(20)

In [ ]:
duplicates = df_all[df_all.duplicated("doc_id", keep=False)]

print("Duplicated doc_id rows:", len(duplicates))

if len(duplicates):
    display(
        duplicates[
            [
                "doc_id",
                "source_file",
            ]
        ]
        .sort_values("doc_id")
        .head(30)
    )

In [ ]:
small_docs = df_all.sort_values("word_count").head(20)

small_docs[
    [
        "doc_id",
        "filename",
        "corpus",
        "source_type",
        "synthetic_type",
        "word_count",
        "char_count",
        "source_file",
    ]
]

In [ ]:
large_docs = df_all.sort_values("char_count", ascending=False).head(20)

large_docs[
    [
        "doc_id",
        "filename",
        "corpus",
        "source_type",
        "synthetic_type",
        "word_count",
        "char_count",
        "source_file",
    ]
]